# Topic 28 — Deep Learning Optimization
### Theory → batch/epoch/iteration → SGD vs Momentum vs Adam from scratch → learning-rate scheduling.

**Vocabulary first:**
- **Batch**: a chunk of training samples processed together before one weight update.
- **Batch size**: how many samples are in each batch.
- **Mini-batch**: a batch that's a small subset of the full dataset (the common case in practice).
- **Epoch**: one full pass through the entire training dataset.
- **Iteration (step)**: one weight update (one batch processed).
  If you have 1000 samples and batch_size=100, that's 10 iterations per epoch.
- **Learning rate**: the step size for each weight update (Topic 4).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)

## 1. Batch vs mini-batch vs stochastic (SGD) gradient descent

- **Batch (full-batch) GD**: use ALL training data to compute each gradient step. Stable but slow
  per step, and can get stuck more easily since every step is identical given the same weights.
- **Stochastic GD (SGD)**: use just ONE random sample per step. Very noisy, but fast per step and
  the noise can actually help escape shallow local minima.
- **Mini-batch GD**: use a small batch (e.g. 32-256 samples) per step — the practical default,
  balancing stability and speed.

In [ ]:
# A toy loss surface: minimize (w-4)^2 using data-point-level noise to simulate SGD
def loss_grad_single_sample(w, noise_scale=2.0):
    true_grad = 2 * (w - 4)
    noisy_grad = true_grad + rng.normal(0, noise_scale)   # SGD sees noisy, per-sample gradients
    return noisy_grad

def run_gd(w_start, lr, steps, batch_type="full", batch_size=1, noise_scale=2.0):
    w = w_start
    history = [w]
    for step in range(steps):
        if batch_type == "full":
            grad = 2 * (w - 4)   # no noise -- exact gradient using "all the data"
        else:
            # average gradient over `batch_size` noisy samples
            grads = [loss_grad_single_sample(w, noise_scale) for _ in range(batch_size)]
            grad = np.mean(grads)
        w -= lr * grad
        history.append(w)
    return history

full_batch_history = run_gd(0.0, lr=0.1, steps=50, batch_type="full")
sgd_history = run_gd(0.0, lr=0.1, steps=50, batch_type="mini", batch_size=1)
minibatch_history = run_gd(0.0, lr=0.1, steps=50, batch_type="mini", batch_size=16)

plt.figure(figsize=(7, 4))
plt.plot(full_batch_history, label="full-batch (smooth, no noise)")
plt.plot(sgd_history, label="SGD, batch_size=1 (noisy)")
plt.plot(minibatch_history, label="mini-batch, batch_size=16 (in between)")
plt.axhline(4, color="gray", linestyle="--", label="true minimum")
plt.xlabel("step"); plt.ylabel("w")
plt.legend()
plt.title("Full-batch vs SGD vs mini-batch convergence paths")
plt.show()

## 2. Momentum

Plain gradient descent can be slow across long, shallow valleys, and can oscillate across steep,
narrow ones. **Momentum** keeps a running average of past gradients (like a rolling ball
building up speed) so it keeps moving in a consistent direction and dampens oscillation.

```text
velocity = beta * velocity + (1 - beta) * gradient
w = w - lr * velocity
```

In [ ]:
def loss_2d(w):   # a deliberately elongated ("narrow valley") loss surface
    return 0.1 * w[0]**2 + 5 * w[1]**2

def grad_2d(w):
    return np.array([0.2 * w[0], 10 * w[1]])

def plain_gd(w_start, lr, steps):
    w = np.array(w_start, dtype=float)
    path = [w.copy()]
    for _ in range(steps):
        w = w - lr * grad_2d(w)
        path.append(w.copy())
    return np.array(path)

def momentum_gd(w_start, lr, steps, beta=0.9):
    w = np.array(w_start, dtype=float)
    velocity = np.zeros_like(w)
    path = [w.copy()]
    for _ in range(steps):
        grad = grad_2d(w)
        velocity = beta * velocity + (1 - beta) * grad
        w = w - lr * velocity
        path.append(w.copy())
    return np.array(path)

start = [8.0, 8.0]
path_plain = plain_gd(start, lr=0.15, steps=40)
path_momentum = momentum_gd(start, lr=0.15, steps=40)

plt.figure(figsize=(6, 5))
plt.plot(path_plain[:, 0], path_plain[:, 1], "o-", label="plain GD", alpha=0.7, markersize=3)
plt.plot(path_momentum[:, 0], path_momentum[:, 1], "o-", label="momentum", alpha=0.7, markersize=3)
plt.scatter([0], [0], c="red", marker="*", s=200, label="minimum")
plt.xlabel("w0"); plt.ylabel("w1")
plt.legend()
plt.title("Plain GD (zig-zags) vs Momentum (smoother, faster convergence)")
plt.show()

## 3. Adam — the modern default

**Adam** (Adaptive Moment Estimation) combines momentum (tracking the gradient's mean, like above)
with ALSO tracking the gradient's variance, and uses both to give every individual parameter its
OWN adaptive effective learning rate. This is why Adam is usually the default optimizer for deep
learning in practice — it needs little tuning and handles poorly-scaled gradients gracefully.

In [ ]:
def adam_gd(w_start, lr, steps, beta1=0.9, beta2=0.999, eps=1e-8):
    w = np.array(w_start, dtype=float)
    m = np.zeros_like(w)   # 1st moment (mean of gradients, like momentum)
    v = np.zeros_like(w)   # 2nd moment (uncentered variance of gradients)
    path = [w.copy()]
    for t in range(1, steps + 1):
        grad = grad_2d(w)
        m = beta1 * m + (1 - beta1) * grad
        v = beta2 * v + (1 - beta2) * (grad ** 2)
        m_hat = m / (1 - beta1 ** t)   # bias correction (important early in training)
        v_hat = v / (1 - beta2 ** t)
        w = w - lr * m_hat / (np.sqrt(v_hat) + eps)
        path.append(w.copy())
    return np.array(path)

path_adam = adam_gd(start, lr=0.5, steps=40)

plt.figure(figsize=(6, 5))
plt.plot(path_plain[:, 0], path_plain[:, 1], "o-", label="plain GD", alpha=0.5, markersize=3)
plt.plot(path_momentum[:, 0], path_momentum[:, 1], "o-", label="momentum", alpha=0.5, markersize=3)
plt.plot(path_adam[:, 0], path_adam[:, 1], "o-", label="Adam", alpha=0.9, markersize=3)
plt.scatter([0], [0], c="red", marker="*", s=200, label="minimum")
plt.legend()
plt.title("Plain GD vs Momentum vs Adam")
plt.show()

## 4. AdamW — Adam with corrected weight decay

**Weight decay** (a form of L2 regularization, previewed here, covered fully in Topic 31) shrinks
weights slightly on every step to fight overfitting. In plain Adam, mixing weight decay directly
into the gradient interacts awkwardly with Adam's adaptive scaling. **AdamW** decouples weight
decay from the gradient-based update, applying it separately and more effectively. In PyTorch
(Topic 29), this is simply `torch.optim.AdamW` instead of `torch.optim.Adam`.

In [ ]:
def adamw_gd(w_start, lr, steps, weight_decay=0.01, beta1=0.9, beta2=0.999, eps=1e-8):
    w = np.array(w_start, dtype=float)
    m = np.zeros_like(w)
    v = np.zeros_like(w)
    path = [w.copy()]
    for t in range(1, steps + 1):
        grad = grad_2d(w)
        m = beta1 * m + (1 - beta1) * grad
        v = beta2 * v + (1 - beta2) * (grad ** 2)
        m_hat = m / (1 - beta1 ** t)
        v_hat = v / (1 - beta2 ** t)
        w = w - lr * (m_hat / (np.sqrt(v_hat) + eps) + weight_decay * w)   # decay applied separately
        path.append(w.copy())
    return np.array(path)

path_adamw = adamw_gd(start, lr=0.5, steps=40)
print("Adam final position:", path_adam[-1])
print("AdamW final position:", path_adamw[-1], "(pulled slightly more toward 0 by weight decay)")

## 5. Learning-rate scheduling

Instead of a fixed learning rate for all of training, gradually REDUCE it — large steps early
(fast progress), small steps late (fine-tune precisely near the minimum without overshooting).

In [ ]:
def step_decay_schedule(initial_lr, epoch, drop_every=10, drop_factor=0.5):
    return initial_lr * (drop_factor ** (epoch // drop_every))

def cosine_schedule(initial_lr, epoch, total_epochs):
    return initial_lr * 0.5 * (1 + np.cos(np.pi * epoch / total_epochs))

epochs = np.arange(50)
step_lrs = [step_decay_schedule(0.1, e) for e in epochs]
cosine_lrs = [cosine_schedule(0.1, e, 50) for e in epochs]

plt.figure(figsize=(6, 4))
plt.plot(epochs, step_lrs, label="step decay")
plt.plot(epochs, cosine_lrs, label="cosine schedule")
plt.xlabel("epoch"); plt.ylabel("learning rate")
plt.legend()
plt.title("Common learning-rate schedules")
plt.show()

## Exercise

In [ ]:
# --- Try it yourself ---
# 1. Increase lr to 0.4 in plain_gd on the narrow-valley loss -- does it diverge? Compare to
#    momentum_gd at the same lr.
# 2. Change beta1/beta2 in adam_gd and see how the convergence path changes.
# 3. Implement a simple "warmup" schedule: linearly increase lr from 0 to 0.1 over the first 5 epochs,
#    then apply step decay afterward.
# 4. In one sentence: why might Adam converge FASTER than plain SGD, but sometimes generalize
#    slightly WORSE on the test set in some published deep learning research? (This is a
#    "just be aware of it" question -- no need to derive it, just note the tradeoff exists.)

---
### Next up: **Topic 29 — PyTorch Fundamentals**.

Say "next" when you're ready.